In [1]:
from dataclasses import dataclass
from contextlib import contextmanager
import portion as P
import interval_logic as il

@dataclass(frozen=True)
class Contract:
    assumption: il.Expression | str
    guarantee: il.Expression | str

    def __post_init__(self) -> None:
        assumption = il.as_expr(self.assumption).simplify()

        # Maintain saturated-form invariant
        guarantee = il.Or((
            il.as_expr(self.guarantee),
            il.Not(assumption),
        )).simplify()

        object.__setattr__(self, "assumption", assumption)
        object.__setattr__(self, "guarantee", guarantee)

    @staticmethod
    def _s(assumption: il.Expression | str, guarantee: il.Expression | str, bid: float | None = None) -> str:
        """Return a string representation of the contract."""
        mid = "\u2223" if bid is None else f"\u2223 (bid: {bid:.2f}) \u2223"
        return f"\u27E8 {il.as_expr(assumption)} {mid} {il.as_expr(guarantee)} \u27E9"

    def __str__(self) -> str:
        return self._s(self.assumption, self.guarantee)

    @staticmethod
    def merge_s(self: 'Contract', other: 'Contract') -> str:
        """Merge two contracts by combining their assumptions and guarantees."""
        return f"{self!s} \u2297 {other!s}"

    def merge(self, other: 'Contract') -> 'Contract':
        """Merge two contracts by combining their assumptions and guarantees."""
        return Contract(il.And((self.assumption, other.assumption)),
                        il.Or((il.And((self.guarantee, other.guarantee)),
                              il.Not(il.And((self.assumption, other.assumption))))))

    @staticmethod
    def separate_s(self, other: 'Contract') -> str:
        """Separate the contract into its assumption and guarantee."""
        return f"{self!s} \u29BC {other!s}"

    def separate(self, other: 'Contract') -> 'Contract':
        """Separate the contract into its assumption and guarantee."""
        return Contract(il.Or((il.And((self.assumption, other.guarantee)),
                               il.Not(il.And((self.guarantee, other.assumption))))),
                        il.And((self.guarantee, other.assumption)))

    def to_nnf(self) -> 'Contract':
        """Convert the contract to Negation Normal Form (NNF)."""
        return Contract(il.to_nnf(self.assumption), il.to_nnf(self.guarantee))
    
    def simplify(self) -> 'Contract':
        """Simplify the contract by removing redundant assumptions and guarantees."""
        return Contract(self.assumption.simplify(), self.guarantee.simplify())

    def replace(self, *fns) -> 'Contract':
        """Replace parts of the contract using a given function."""
        return Contract(self.assumption.replace(*fns), self.guarantee.replace(*fns))

    def is_consistent(self, universe: P.Interval) -> bool:
        """Check if the contract is consistent (i.e., the assumption implies the guarantee)."""
        if self.guarantee == il.Bool(True):
            return True
        return il.denote(il.And((self.assumption, self.guarantee)),
                         {}, universe) == universe

@contextmanager
def step(msg: str, disable: int = 4):

    counter = list(getattr(step, "counter", (0,)))
    counter[-1] += 1
    if len(counter) == 1:
        println(f"⏺ Step {counter[0]}: {msg}")
    else:
        println(f"⏺ Step {'.'.join(map(str, counter))}: {msg}", 
                marker=["├", "│"])
    step.counter = tuple(counter + [0])
    _disable = getattr(step, "disable", 4)
    step.disable = disable
    try:
        yield
    finally:
        pass
    step.disable = _disable
    step.counter = step.counter[:-1]
    if len(step.counter) == 1:
        println('╯', left=0)
    else:
        println('', marker=["├─╯", "│"])

def reset_step():
    step.counter = (0,)
    step.disable = 4

def println(*lines: str, **kwds):
    assert all(isinstance(line, str) for line in lines)
    lines = [""] if len(lines) == 0 else list(lines)
    marker = kwds.get("marker", "│")
    top = kwds.get("top", kwds.get("y", 0))
    bottom = kwds.get("bottom", kwds.get("y", 0))
    left = kwds.get("left", kwds.get("x", 0))
    right = kwds.get("right", kwds.get("x", 0))

    indent = len(getattr(step, "counter", (0,))) - 1
    if indent >= getattr(step, "disable", 4):
        return
    ms = [marker] if isinstance(marker, str) else list(marker)
    ms = [m + " " for m in ms]
    ms += [ms[-1]] * (indent - len(ms)) # may be more than `indent` markers
    ws = "".join(reversed(ms[:indent]))

    max_width = max(len(line) for line in lines) + left + right
    lines = [ws + (" " * left + line + " " * right).ljust(max_width)
             for line in lines]
    lines = [ws]*top + lines + [ws]*bottom

    print("\n".join(lines))

def printeq(arg, *args):
    lines = [str(arg)]
    lines += [f"= {arg!s}" for arg in args]
    println(*lines)

with step("Test"):
    println("This is a test.")
    with step("Subtest"):
        println("Hello, world!")
        printeq(1 + 2, 3, 4)
    println("Back to the main test.")
    with step("Another subtest"):
        println("Goodbye, world!")

⏺ Step 1: Test
│ This is a test.
├ ⏺ Step 1.1: Subtest
│ │ Hello, world!
│ │ 3  
│ │ = 3
│ │ = 4
├─╯ 
│ Back to the main test.
├ ⏺ Step 1.2: Another subtest
│ │ Goodbye, world!
├─╯ 
╯


In [2]:
def keep(n):
    j = il.Signal('j')
    n = il.Natural(n)
    p = il.Atom('p')
    q = il.Atom('q')
    def replacer(psi):
        match psi:
            case il.Equal(left, right):
                if left == j:
                    return  p if right == n else q
                if right == j:
                    return p if left == n else q
        return psi
    return replacer

def weaken(psi):
    match psi:
        case il.ForAll(I, p):
            return il.Exists(I, p)
    return psi

def strengthen(psi):
    match psi:
        case il.Exists(I, p):
            return il.ForAll(I, p)
    return psi


def overlap(psi):
    subst = False
    match psi:
        case il.Or((left, right)):
            match (left, right):
                case (il.ForAll(I, p), il.Exists(J, q)):
                    if p == il.Not(q): subst = True
                case (il.Exists(J, q), il.ForAll(I, p)):
                    if p == il.Not(q): subst = True
            return il.Or((
                il.ForAll(I - J, p),
                il.Exists(J, q),
            )) if subst else psi
        case il.And((left, right)):
            match (left, right):
                case (il.ForAll(I, p), il.Exists(J, q)):
                    if p == il.Not(q): subst = True
                case (il.Exists(J, q), il.ForAll(I, p)):
                    if p == il.Not(q): subst = True
            return il.And((
                il.ForAll(I, p),
                il.Exists(J - I, q),
            )) if subst else psi
    return psi

In [ ]:
with step('Protocol'):

    println('Commitments')
    printeq(C_cmt := Contract(True, '∀[1,3]:j=1'),
            C_cmt := C_cmt.replace(keep(2)))

    println('Request', top=1)
    printeq(C_req := Contract('m', '∀[2,4]:j=2'),
            C_req := C_req.replace(keep(2)))

    println('Weakened Request', top=1)
    printeq(C_req_w := C_req.replace(weaken))

    println('Separated', top=1)
    printeq(Contract.separate_s(C_req_w, C_cmt),
            C_s := C_req_w.separate(C_cmt))

    println('Merged Separated', top=1)
    printeq(Contract.merge_s(C_s, C_cmt),
            C_ms := C_s.merge(C_cmt))

    println('Postprocess', top=1)
    printeq(Contract._s(True, C_ms.guarantee & C_ms.assumption),
            C_post := Contract(True, C_ms.guarantee & C_ms.assumption),
            C_post := C_post.replace(lambda psi: il.as_expr('¬p') if psi == il.Atom('q') else psi),
            C_post := C_post.replace(overlap),
            )

├ ⏺ Step 2.1: Protocol
│ │ Commitments


FrozenInstanceError: cannot assign to field 'assumption'

In [ ]:
@dataclass(frozen=True)
class Request:
    job: int
    energy: float
    interval: P.Interval
    price: float

    def to_contract(self) -> Contract:
        return Contract(True, il.ForAll(self.interval, il.Signal("j").equals(self.job)))

@dataclass
class NegotiationState:
    expected_utility: float
    min_utility: float
    deadline: int
    lambda_: float

    @staticmethod
    def concede(U, t, tau, lam):
        return U - U * (t / tau) ** lam

⟨ ⊤ ∣ (bid: 2.00) ∣ m ∧ (∀[1.0,3.0]: ¬p) ∧ (∃[2.0,4.0]: p) ⟩
